# SO(3) covariance tuning

A small differentiable InEKF: first verify a two-parameter scalar graph against finite differences, then learn diagonal Q/R with the same equations in Torch. The CUDA path captures the static forward + backward graph; optimizer updates remain explicit.

In [ ]:
import math, random

class _SgScalar:
    """One node in a scalar computation graph."""
    def __init__(self, data, children=()):
        self.data, self.grad = data, 0.0
        self._backward, self._prev = (lambda: None), set(children)

    def __add__(self, o):
        o = o if isinstance(o, _SgScalar) else _SgScalar(o)
        out = _SgScalar(self.data + o.data, (self, o))
        def bw():
            self.grad += out.grad; o.grad += out.grad
        out._backward = bw
        return out

    def __mul__(self, o):
        o = o if isinstance(o, _SgScalar) else _SgScalar(o)
        out = _SgScalar(self.data * o.data, (self, o))
        def bw():
            self.grad += o.data * out.grad; o.grad += self.data * out.grad
        out._backward = bw
        return out

    def __pow__(self, k):
        out = _SgScalar(self.data ** k, (self,))
        def bw():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = bw
        return out

    def _unary(self, y, dydx):
        out = _SgScalar(y, (self,))
        def bw():
            self.grad += dydx * out.grad
        out._backward = bw
        return out

    def sin(self):  return self._unary(math.sin(self.data), math.cos(self.data))
    def cos(self):  return self._unary(math.cos(self.data), -math.sin(self.data))
    def exp(self):  y = math.exp(self.data);  return self._unary(y, y)
    def log(self):  return self._unary(math.log(self.data), 1.0 / self.data)
    def sqrt(self): y = math.sqrt(self.data); return self._unary(y, 0.5 / y)

    def backward(self):
        topo, seen = [], set()
        def build(v):
            if v not in seen:
                seen.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo): v._backward()

    __neg__  = lambda s: s * -1.0
    __radd__ = lambda s, o: s + o
    __rmul__ = lambda s, o: s * o
    __sub__  = lambda s, o: s + (-o)
    __rsub__ = lambda s, o: (-s) + o
    __truediv__ = lambda s, o: s * o ** -1

def _sg_atan2(y, x):
    out = _SgScalar(math.atan2(y.data, x.data), (y, x))
    def bw():
        d = x.data * x.data + y.data * y.data
        y.grad += (x.data / d) * out.grad
        x.grad += (-y.data / d) * out.grad
    out._backward = bw
    return out

# 3x3 helpers over lists of _SgScalar.
_sg_V   = lambda x: x if isinstance(x, _SgScalar) else _SgScalar(x)
_sg_eye = lambda: [[_sg_V(1.0 if i == j else 0.0) for j in range(3)] for i in range(3)]
_sg_hat = lambda w: [[_sg_V(0.0), -w[2], w[1]],
                     [w[2], _sg_V(0.0), -w[0]],
                     [-w[1], w[0], _sg_V(0.0)]]
_sg_mt  = lambda A: [[A[j][i] for j in range(3)] for i in range(3)]
_sg_mm  = lambda A, B: [[sum((A[i][k] * B[k][j] for k in range(3)), _sg_V(0.0))
                         for j in range(3)] for i in range(3)]
_sg_mv  = lambda A, x: [sum((A[i][k] * x[k] for k in range(3)), _sg_V(0.0)) for i in range(3)]
_sg_madd  = lambda A, B: [[A[i][j] + B[i][j] for j in range(3)] for i in range(3)]
_sg_msub  = lambda A, B: [[A[i][j] - B[i][j] for j in range(3)] for i in range(3)]
_sg_scale = lambda A, s: [[A[i][j] * s for j in range(3)] for i in range(3)]

def _sg_inv3(M):
    (a, b, c), (d, e, f), (g, h, i) = M
    A, B, C = e*i - f*h, f*g - d*i, d*h - e*g
    D, E, F = c*h - b*i, a*i - c*g, b*g - a*h
    G, H, I = b*f - c*e, c*d - a*f, a*e - b*d
    idet = (a*A + b*B + c*C) ** -1
    return [[A*idet, D*idet, G*idet],
            [B*idet, E*idet, H*idet],
            [C*idet, F*idet, I*idet]]

def _sg_so3_exp(phi):
    th = (phi[0]*phi[0] + phi[1]*phi[1] + phi[2]*phi[2] + 1e-24).sqrt()
    K = _sg_hat(phi)
    return _sg_madd(_sg_madd(_sg_eye(), _sg_scale(K, th.sin() / th)),
                    _sg_scale(_sg_mm(K, K), (1 - th.cos()) / (th * th)))

def _sg_so3_log(R):
    cos_th = (R[0][0] + R[1][1] + R[2][2] - 1) * 0.5
    w = [R[2][1] - R[1][2], R[0][2] - R[2][0], R[1][0] - R[0][1]]
    n = (w[0]*w[0] + w[1]*w[1] + w[2]*w[2] + 1e-24).sqrt()  # 2 sin(theta)
    s = _sg_atan2(n * 0.5, cos_th) / n
    return [wi * s for wi in w]

In [ ]:
# Tiny T=5 dataset: constant angular rate, noisy gyro, noisy attitude measurements.
random.seed(1)
_sg_T, _sg_dt = 5, 0.05
_sg_w_true, _sg_gyro_sig, _sg_meas_sig = [0.3, -0.2, 0.15], 0.02, 0.05

_sg_rotf = lambda phi: [[e.data for e in row] for row in _sg_so3_exp([_sg_V(p) for p in phi])]
_sg_fmm  = lambda A, B: [[sum(A[i][k]*B[k][j] for k in range(3)) for j in range(3)] for i in range(3)]

_sg_Rgt, _sg_gyro, _sg_Rmeas = [[[1.0,0,0],[0,1.0,0],[0,0,1.0]]], [], []
for _k in range(_sg_T):
    _sg_gyro.append([w + random.gauss(0, _sg_gyro_sig) for w in _sg_w_true])
    _sg_Rmeas.append(_sg_fmm(_sg_Rgt[_k], _sg_rotf([random.gauss(0, _sg_meas_sig) for _ in range(3)])))
    if _k < _sg_T - 1:
        _sg_Rgt.append(_sg_fmm(_sg_Rgt[_k], _sg_rotf([w * _sg_dt for w in _sg_w_true])))

def _sg_forward(raw_q0, raw_r0):
    # Two scalars -> isotropic Q/R; softplus keeps them positive.
    sp = lambda x: (1 + x.exp()).log()
    q, r = sp(raw_q0) + 1e-6, sp(raw_r0) + 1e-6
    LM = lambda M: [[_sg_V(M[i][j]) for j in range(3)] for i in range(3)]
    Q  = [[q if i == j else _sg_V(0.0) for j in range(3)] for i in range(3)]
    Rc = [[r if i == j else _sg_V(0.0) for j in range(3)] for i in range(3)]
    Rhat, P, loss = LM(_sg_Rmeas[0]), _sg_scale(_sg_eye(), _sg_V(0.01)), _sg_V(0.0)
    for k in range(_sg_T):
        w  = [_sg_V(x) for x in _sg_gyro[k]]
        Rp = _sg_mm(Rhat, _sg_so3_exp([wi * _sg_dt for wi in w]))   # propagate
        F  = _sg_so3_exp([-wi * _sg_dt for wi in w])
        Pp = _sg_madd(_sg_mm(_sg_mm(F, P), _sg_mt(F)), Q)
        res = _sg_so3_log(_sg_mm(_sg_mt(Rp), LM(_sg_Rmeas[k])))     # update
        K  = _sg_mm(Pp, _sg_inv3(_sg_madd(Pp, Rc)))
        Rhat = _sg_mm(Rp, _sg_so3_exp(_sg_mv(K, res)))
        IK = _sg_msub(_sg_eye(), K)                                 # Joseph form
        P  = _sg_madd(_sg_mm(_sg_mm(IK, Pp), _sg_mt(IK)),
                      _sg_mm(_sg_mm(K, Rc), _sg_mt(K)))
        e  = _sg_so3_log(_sg_mm(_sg_mt(LM(_sg_Rgt[k])), Rhat))      # attitude error
        loss = loss + e[0]*e[0] + e[1]*e[1] + e[2]*e[2]
    return loss * (1.0 / _sg_T)

_sg_raw_q0, _sg_raw_r0 = _SgScalar(1.0), _SgScalar(-2.0)  # Q too big, R too small
_sg_loss = _sg_forward(_sg_raw_q0, _sg_raw_r0)
_sg_loss.backward()

_sg_h = 1e-5
_sg_fd = lambda dq, dr: (_sg_forward(_SgScalar(1.0 + dq), _SgScalar(-2.0 + dr)).data
                         - _sg_forward(_SgScalar(1.0 - dq), _SgScalar(-2.0 - dr)).data) / (2 * _sg_h)
_sg_fd_q, _sg_fd_r = _sg_fd(_sg_h, 0.0), _sg_fd(0.0, _sg_h)

print(f"loss={_sg_loss.data:.6f}")
print(f"raw_q0: graph_grad={_sg_raw_q0.grad:+.8f}  finite_diff={_sg_fd_q:+.8f}  abs_err={abs(_sg_raw_q0.grad - _sg_fd_q):.2e}")
print(f"raw_r0: graph_grad={_sg_raw_r0.grad:+.8f}  finite_diff={_sg_fd_r:+.8f}  abs_err={abs(_sg_raw_r0.grad - _sg_fd_r):.2e}")
assert abs(_sg_raw_q0.grad - _sg_fd_q) < 1e-4 and abs(_sg_raw_r0.grad - _sg_fd_r) < 1e-4
print("scalar graph check passed")

## Torch version

The tensor model keeps the propagation, innovation, solve, Joseph update, and trajectory loss differentiable. Static data and shapes also make one complete training step CUDA-Graph friendly.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, "| torch", torch.__version__)

def hat(w):
    # w: (..., 3) -> (..., 3, 3)
    z = torch.zeros_like(w[..., 0])
    r0 = torch.stack([z, -w[..., 2], w[..., 1]], dim=-1)
    r1 = torch.stack([w[..., 2], z, -w[..., 0]], dim=-1)
    r2 = torch.stack([-w[..., 1], w[..., 0], z], dim=-1)
    return torch.stack([r0, r1, r2], dim=-2)


def vee(W):
    return torch.stack([W[..., 2, 1], W[..., 0, 2], W[..., 1, 0]], dim=-1)


def so3_exp(phi):
    th = torch.sqrt((phi * phi).sum(-1, keepdim=True) + 1e-24)
    K = hat(phi)
    a = (torch.sin(th) / th)[..., None]
    b = ((1 - torch.cos(th)) / (th * th))[..., None]
    I = torch.eye(3, dtype=phi.dtype, device=phi.device)
    return I + a * K + b * (K @ K)


def so3_log(R):
    tr = R[..., 0, 0] + R[..., 1, 1] + R[..., 2, 2]
    cos = ((tr - 1) * 0.5).clamp(-1.0, 1.0)
    w = vee(R - R.transpose(-1, -2))
    sin = torch.linalg.norm(w, dim=-1) * 0.5
    th = torch.atan2(sin, cos)
    scale = th / (2 * sin + 1e-24)
    return scale[..., None] * w



In [ ]:
def make_data(T=200, dt=0.02, gyro_sigma=0.01, meas_sigma=0.05, seed=0):
    gen = torch.Generator().manual_seed(seed)
    t = torch.arange(T) * dt
    omega_true = torch.stack([
        0.6 * torch.sin(0.7 * t),
        0.5 * torch.cos(0.5 * t),
        0.4 * torch.sin(0.3 * t + 0.5),
    ], dim=-1)

    Rgt = torch.zeros(T, 3, 3)
    Rgt[0] = torch.eye(3)
    for k in range(T - 1):
        Rgt[k + 1] = Rgt[k] @ so3_exp(omega_true[k] * dt)

    gyro = omega_true + gyro_sigma * torch.randn(T, 3, generator=gen)
    Rmeas = Rgt @ so3_exp(meas_sigma * torch.randn(T, 3, generator=gen))
    return omega_true, Rgt, gyro, Rmeas


T, dt = 120, 0.02
gyro_sigma, meas_sigma = 0.01, 0.05
omega_true, Rgt, gyro, Rmeas = make_data(T, dt, gyro_sigma, meas_sigma, seed=0)
Rgt, gyro, Rmeas = Rgt.to(device), gyro.to(device), Rmeas.to(device)
data = (gyro, Rmeas, Rgt)


In [ ]:
class SO3InEKF(nn.Module):
    def __init__(self, raw_q_init, raw_r_init, eps=1e-6):
        super().__init__()
        self.raw_q = nn.Parameter(torch.tensor(raw_q_init))
        self.raw_r = nn.Parameter(torch.tensor(raw_r_init))
        self.eps = eps

    def covs(self):
        q = F.softplus(self.raw_q) + self.eps
        r = F.softplus(self.raw_r) + self.eps
        return torch.diag(q), torch.diag(r)

    def forward(self, gyro, Rmeas, dt):
        Q, Rc = self.covs()
        I = torch.eye(3, dtype=gyro.dtype, device=gyro.device)
        Rhat = Rmeas[0].clone()
        P = 0.01 * I
        out = []
        for k in range(gyro.shape[0]):
            Rp = Rhat @ so3_exp(gyro[k] * dt)
            Fk = so3_exp(-gyro[k] * dt)
            Pp = Fk @ P @ Fk.T + Q

            res = so3_log(Rp.T @ Rmeas[k])
            S = Pp + Rc
            L, info = torch.linalg.cholesky_ex(S, check_errors=False)
            K = torch.cholesky_solve(Pp.T, L).T
            Rhat = Rp @ so3_exp(K @ res)
            IK = I - K
            P = IK @ Pp @ IK.T + K @ Rc @ K.T
            out.append(Rhat)
        return torch.stack(out)


def loss_fn(Rhat, Rgt):
    e = so3_log(Rgt.transpose(-1, -2) @ Rhat)
    return (e * e).sum(-1).mean()


In [ ]:
raw_q_init = [1.0, 1.0, 1.0]
raw_r_init = [-3.0, -3.0, -3.0]

m0 = SO3InEKF(raw_q_init, raw_r_init).to(device)
Q0, R0 = m0.covs()

class CapturedBackward:
    """Replay a static forward + backward graph; update parameters outside it."""

    def __init__(self, model):
        self.model = model
        self.params = list(model.parameters())
        self.loss = torch.zeros((), device=device)
        current = torch.cuda.current_stream()
        side = torch.cuda.Stream()
        side.wait_stream(current)
        with torch.cuda.stream(side):
            for _ in range(3):
                model.zero_grad(set_to_none=True)
                loss_fn(model(gyro, Rmeas, dt), Rgt).backward()
        current.wait_stream(side)
        model.zero_grad(set_to_none=True)
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph, stream=side):
            loss = loss_fn(model(gyro, Rmeas, dt), Rgt)
            loss.backward()
            self.loss.copy_(loss.detach())
        current.wait_stream(side)
        assert all(p.grad is not None for p in self.params)

    def replay(self):
        self.model.zero_grad(set_to_none=False)
        self.graph.replay()
        return self.loss


def train(opt_kind, lr, steps=160):
    torch.manual_seed(0)
    model = SO3InEKF(raw_q_init, raw_r_init).to(device)
    opt = (torch.optim.Adam(model.parameters(), lr=lr) if opt_kind == "adam"
           else torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9))
    captured = CapturedBackward(model) if device == "cuda" else None
    if device == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    losses = []
    for _ in range(steps):
        if captured is None:
            opt.zero_grad(set_to_none=True)
            loss = loss_fn(model(gyro, Rmeas, dt), Rgt)
            loss.backward()
        else:
            loss = captured.replay()
        opt.step()
        losses.append(loss.detach().clone())
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    losses = torch.stack(losses).cpu().tolist()
    print(f"{opt_kind}: {losses[0]:.3e} -> {losses[-1]:.3e} in {elapsed:.2f}s")
    return model, losses


In [ ]:
model_adam, loss_adam = train("adam", 3e-2)
model_sgd, loss_sgd = train("sgd", 3.0)


In [ ]:
with torch.no_grad():
    err_init = so3_log(Rgt.transpose(-1, -2) @ m0(gyro, Rmeas, dt)).norm(dim=-1)
    err_adam = so3_log(Rgt.transpose(-1, -2) @ model_adam(gyro, Rmeas, dt)).norm(dim=-1)
    Qa, Ra = model_adam.covs()

fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].semilogy(loss_adam, label="Adam")
ax[0].semilogy(loss_sgd, label="SGD + momentum")
ax[0].set(xlabel="step", ylabel="loss", title="training")
ax[0].legend(); ax[0].grid(True, alpha=0.3)
ax[1].plot(err_init.cpu(), label="initial", alpha=0.7)
ax[1].plot(err_adam.cpu(), label="trained")
ax[1].set(xlabel="time step", ylabel="error [rad]", title="attitude error")
ax[1].legend(); ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print("Adam sqrt diagQ:", Qa.diag().sqrt().cpu().numpy().round(4))
print("Adam sqrt diagR:", Ra.diag().sqrt().cpu().numpy().round(4))


Adam is easier to tune here; SGD is more learning-rate sensitive. The learned Q/R minimize trajectory error and need not recover the data-generation noise exactly.